# Raw data bias check (before cleaning)
2026 Joey Manani & Anchorfish Team

Looks at the raw dataset file **before** cleaning

A big gap between features is **not** on its own a reason to drop something. A good feature is supposed to differ between type. 

Let's give them named (these are invented here, not actual terms):
1. **Same-site test:** is the feature present on the same site with and without the feature? Clean it out of the URL.
2. **Reality test:** is the rate believable for real traffic? Is it a reasonable value (human judgment)
3. **Research test:** is it consistent with actual research?

We will explain more these tests in the report, but same-site is quite important to understand, so I'll (Joey) provide a brief description:

Basically speaking, will the feature change depending on HOW the URL was actually stored? i.e from the set, is "http://www.phishingurl.com/" technically "worse" (phishing score) than "phishingurl.com"? These ARE the same site, so the feature should not be influenced by the way it's formatted in the URL.

In [ ]:
import sys; print(sys.executable)

from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

DATA = Path("malicious_phish.csv") # provided set
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

SCHEME = r"^[a-zA-Z][a-zA-Z0-9+.-]*://"  # http://, https://

df = pd.read_csv(DATA)
df.type.value_counts()

## 1. URL formatting by type
Shown as **% of each type**, not numbers: there are about 5x more legit URLs so we gotta scale them accordingly.

In [ ]:
bare = df.url.str.replace(SCHEME, "", regex=True)  # so "http://www.x.com" still counts as starting with www


flags = pd.DataFrame({
    "type": df.type,
    "has scheme": df.url.str.match(SCHEME),
    "starts www.": bare.str.match(r"(?i)www\d*\."),
    "IP host": bare.str.match(r"(\d{1,3}\.){3}\d{1,3}"), # ipv4
})

# based on the flags, calculate percentages of the totals that have these features, and group these by their URL type
percentage_table = flags.groupby("type").mean() * 100

# must be transposed for plotting
percentage_table.T.plot.bar(rot=0, ylabel="% of URLs in type", title="Formatting by type (raw)")
plt.savefig(FIGURES / "raw_flags_by_type.png")
plt.show()
percentage_table.round(2)

So according to the output above, only 8% of the benigns have a scheme, compared to 26% of the phishing URLs. This means that the presence of a URL scheme is a strong indicator of whether a URL is malicious or not, and to me, a human, this is unreliable therefore we should consider other features for classification.

The same pattern is observed for the other feature including whether it starts with www. Most phishing URLs have a www, but not most of the benigns, meaning a trained model will pick up that anything with www probably means phishing. This is unreliable and must be considered i.e., by less weighting or by dropping this feature (we will drop it).

IP host is interesting. 0.01% of the benigns are an IPv4 host compared to 0.35% of the phishing. This makes sense as a human since most real actual websites have a proper domain instead of just an IP. Malware is even better, where 36% of them are IP-based hosts which makes sense since purchasing a domain 1. costs money, 2. has an ownership record to make it traceable. 

## 2. URL length, legit vs phishing

In [ ]:
length = df.url.str.len()

for type in ["benign", "phishing"]:
    plt.hist(length[df.type == type], 
             bins=range(0, 200, 1), 
             density=True,  # want density not number of
             alpha=0.5, 
             label=type)


plt.xlabel("URL length (characters, limit @ 200)")
plt.ylabel("Density (*100)")
plt.title("URL length by type (raw)")
plt.legend()
plt.savefig(FIGURES / "raw_length_by_type.png")
plt.show()
length.groupby(df.type).median()

### Certain things to play with

- Can set the upper value to 500, but above 200 is mostly outliers. 200 is fine, step = 1. This shows a pretty wide distribution of the URL lengths and is cool to see most is within a certain range.
- If set density to False, look at the massive spike at about 32 (benign), this appears to be a bunch of legitimate IMDB (movie database) URLs. Cool
- The same large spike at phishing url's 27 length seems to be genuine random phishing. They seem to bunch at 27... Wow.

### Interestingly...

So according to this set of data, actually, the PHISHING URLs are shorter than the benign ones. This seems counterintuitive at first glance, but it could be due to the way the dataset was created. This is really interesting to know, and will therefore mean that URL length could be a useful feature for distinguishing between benign and phishing URLs - but we need to be careful not to overfit to this single feature since clearly the counterintuitiveness means it could be either way. This histogram revealed a lot actually. (I, Joey, also played a significant amount using Excel's inbuilt sort and filter functions - we need to do this programmatically as well, therefore this notebook exists)

In [ ]:
feats = pd.read_csv("../features.csv", usecols=["type", "tld"])

# % of each type's URLs per TLD (no-TLD URLs, e.g. IP hosts, left out)
pct = pd.crosstab(feats.tld, feats.type, normalize="columns") * 100
top = pct[(pct.rank(ascending=False) <= 5).any(axis=1)]  # TLDs in the top 5 of at least one type

top.plot.bar(figsize=(12, 4), rot=0, xlabel="", ylabel="% of URLs in type", title="Top 5 TLDs by type")
plt.savefig(FIGURES / "tld_top5_by_type.png")
plt.show()
top.round(2)